[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-labs/blob/main/Module1/Week1_TechStack/Lab1_FirstCommit/notebooks/Lab1_A_Weather_Starter.ipynb)

# Lab 1 - Partner A: the weather half

**You own this notebook and `src/clean_weather.py`. Partner B never touches either one.**

Your job tonight:
1. Load the raw airport observations for **your campus**
2. Turn them into a tidy **hourly** table
3. Make one plot
4. Save `data/clean/weather_hourly.csv` and get it into the repo on branch `dev-weather`

You are **not** doing the demand side, and you are not doing the join. You physically cannot
finish the story alone - that is the point of the lab.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Where the class data lives on GitHub. "raw" = give me the file, not the web page.
RAW = ("https://raw.githubusercontent.com/drdave-teaching/OPIM5512-labs/main/"
       "Module1/Week1_TechStack/Lab1_FirstCommit/data")

# CHANGE THIS to your campus: "hartford" (KBDL) or "stamford" (KBDR)
CAMPUS  = "hartford"
STATION = {"hartford": "KBDL", "stamford": "KBDR"}[CAMPUS]

print("Campus:", CAMPUS, "| Airport:", STATION)

## 1. Look before you parse

Never call `read_csv` on a file you have not looked at. Print the first few raw lines first.

In [ ]:
import urllib.request
url = f"{RAW}/{CAMPUS}/airport_{STATION}_hourly_raw.csv"
print(urllib.request.urlopen(url).read(400).decode())

## 2. Load it

These are **METAR** observations - the same reports pilots read. A few things about the file:

- Missing values are the letter **`M`**, not blanks. Pandas will happily make your whole
  temperature column a *string* because of them.
- Trace precipitation is **`T`**.
- The `valid` column is local time, and observations land at **:51 or :52 past the hour** -
  not on the hour.

| Column | Meaning |
|---|---|
| `station` | airport identifier |
| `valid` | observation time (local) |
| `tmpf` | air temperature, deg F |
| `dwpf` | dew point, deg F |
| `relh` | relative humidity, % |
| `sknt` | wind speed, knots |
| `drct` | wind direction, degrees |
| `p01i` | 1-hour precipitation, inches |
| `vsby` | visibility, miles |
| `skyc1` | sky cover code |

In [ ]:
# TODO: read the CSV. Make sure "M" becomes NaN, not the string "M".
# Hint: pd.read_csv(url, na_values=[...])
wx = ...

print(wx.shape)
wx.head()

In [ ]:
# TODO: check your work. If tmpf is dtype "object" you have a problem.
wx.dtypes

## 3. Make it hourly

Right now your timestamps are 00:51, 01:51, 02:52... To join this to *anything* later, you
need a clean hourly key.

Round **down** to the hour (`.dt.floor("h")`), not to the nearest hour. The 6:51 observation
describes the 6 o'clock hour.

Some hours have two observations (a "special" report filed when weather changes fast) and some
have none. Group by the hour and take the mean - then check whether you have exactly 744 rows.

In [ ]:
# TODO
wx["hour"] = ...

hourly = (wx.groupby("hour", as_index=False)
            .agg(temp_f=("tmpf", "mean"),
                 dewpoint_f=("dwpf", "mean"),
                 humidity_pct=("relh", "mean"),
                 wind_kt=("sknt", "mean")))

print(len(hourly), "hours")   # July 20 - Aug 19 is 744 hours. Do you have all of them?
hourly.head()

In [ ]:
# TODO: how many hours are missing entirely, and how many have a missing temperature?
# This is a real answer you will write into the README data dictionary.


## 4. One plot

Plot temperature over the month. Look at it. Was there a heat wave? When?

In [ ]:
# TODO


## 5. Save it and get it into the repo

**This is the part that matters tonight.** Colab's disk is temporary - the file below
disappears when the runtime disconnects. Getting it into the repo is a deliberate act.

In [ ]:
hourly.to_csv("weather_hourly.csv", index=False)

from google.colab import files
files.download("weather_hourly.csv")

Now, off Colab:

1. Move the downloaded file into your repo folder as `data/clean/weather_hourly.csv`
2. **GitHub Desktop** - you should see it under **Changes**. Confirm you are on branch
   **`dev-weather`** (top of the window!), commit with a real message, then **Push**
3. Back here: **File > Save a copy in GitHub** - repo, branch **`dev-weather`**,
   path `notebooks/weather_eda.ipynb`
4. On github.com: **Compare & pull request**. Ask Partner B to review it.

> If GitHub Desktop says you are on `main`, stop and switch branches first. `main` is
> protected and will reject the push - which is the protection working, not a bug.

### Then write your half of the data dictionary
In the repo `README.md`, document **every column** in `weather_hourly.csv`: name, meaning,
units, and what a missing value means. Partner B is documenting theirs at the same time, in
the same file. Yes, that is going to collide. Read the lab README about that.